In [2]:
# CLEANED — safer, idempotent data preprocessing
import sys, os
import pandas as pd
from sklearn.model_selection import train_test_split

project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

# Load (do not parse dates yet so we can inspect columns reliably)
raw_path = 'data/raw/master.csv'
data = pd.read_csv(raw_path)

# 1) Normalize column names (strip whitespace, lowercase optional)
data.columns = [c.strip() for c in data.columns]
print("\n=== COLUMNS FOUND AFTER STRIP ===")
print(data.columns.tolist())
print("=================================\n")

# 2) If Time (UTC) present — parse it and rename to Date (UTC)
if 'Time (UTC)' in data.columns:
    data['Date'] = pd.to_datetime(data['Time (UTC)'], utc=True)
    # optionally drop original
    # data.drop(columns=['Time (UTC)'], inplace=True)
elif 'Date' in data.columns:
    data['Date'] = pd.to_datetime(data['Date'], utc=True)
else:
    print("Warning: no Time column found — ensure Date exists.")

# --- Indicators (same as you had) ---
data['MA10'] = data['Close'].rolling(10).mean()
data['MA20'] = data['Close'].rolling(20).mean()
data['MA50'] = data['Close'].rolling(50).mean()
data['EMA10'] = data['Close'].ewm(span=10, adjust=False).mean()
data['EMA20'] = data['Close'].ewm(span=20, adjust=False).mean()
data['Volatility'] = data['Close'].rolling(20).std()

delta = data['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss.replace(0, pd.NA)
data['RSI'] = 100 - (100 / (1 + rs))

data['Close_lag1'] = data['Close'].shift(1)
data['Close_lag2'] = data['Close'].shift(2)
data['Close_lag3'] = data['Close'].shift(3)

# Drop rows with NaNs introduced by indicators
data = data.dropna().reset_index(drop=True)

# Create signals
data['Signal'] = 0
data.loc[data['Close'] > data['Close_lag1'], 'Signal'] = 1
data.loc[data['Close'] < data['Close_lag1'], 'Signal'] = -1

# Feature columns — use cleaned column names ('Volume' now)
feature_cols = [
    'Open','High','Low','Close','Volume',
    'MA10','MA20','MA50','EMA10','EMA20',
    'Volatility','RSI','Close_lag1','Close_lag2','Close_lag3'
]

# Validate feature existence
missing = [c for c in feature_cols if c not in data.columns]
if missing:
    raise KeyError(f"Missing feature columns in data: {missing}")

features = data[feature_cols]
target = data['Signal']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, shuffle=False)
print(f"Training rows: {len(X_train)}, Testing rows: {len(X_test)}")

# Simple rule-based evaluation for sanity
def generate_signals(df):
    signals = []
    for _, r in df.iterrows():
        if r['Close'] > r['Close_lag1']:
            signals.append(1)
        elif r['Close'] < r['Close_lag1']:
            signals.append(-1)
        else:
            signals.append(0)
    return signals

train_signals = generate_signals(X_train)
test_signals = generate_signals(X_test)

def evaluate(pred, actual):
    correct = sum(1 for p, a in zip(pred, actual) if p == a)
    return correct / len(actual) * 100

print("Training accuracy: {:.2f}%".format(evaluate(train_signals, y_train.tolist())))
print("Testing accuracy:  {:.2f}%".format(evaluate(test_signals, y_test.tolist())))

# Optional: save cleaned file for later use by training pipeline
clean_path = 'data/raw/master_clean.csv'
data.to_csv(clean_path, index=False)
print(f"Saved cleaned data to {clean_path}")

print("\nFEATURE PROCESSING COMPLETED SUCCESSFULLY.")

# run_phase6.py

# Ensure project root is in sys.path
import sys, os
project_root = os.path.abspath(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)

# Import main_loop
from bot.phase6_bot import main_loop

# Start bot
import pandas as pd

df = pd.read_csv("data/raw/master_clean.csv")
print(df.shape)
print(df.head())
main_loop()


=== COLUMNS FOUND AFTER STRIP ===
['Time (UTC)', 'Open', 'High', 'Low', 'Close', 'Volume']

Training rows: 360192, Testing rows: 90049
Training accuracy: 100.00%
Testing accuracy:  100.00%
Saved cleaned data to data/raw/master_clean.csv

FEATURE PROCESSING COMPLETED SUCCESSFULLY.
No model found at models/model_latest.joblib, running in dynamic mode.


2025-12-03 16:33:55,736 INFO: Connected to MT5 account 297307412
2025-12-03 16:33:55,738 WARNING: Model not found or empty at models/model_latest.joblib
2025-12-03 16:33:55,757 INFO: Disconnected MT5


(450241, 18)
            Time (UTC)     Open     High      Low    Close  Volume  \
0  2019.11.25 02:05:00  108.768  108.768  108.736  108.748  301.93   
1  2019.11.25 02:10:00  108.750  108.762  108.748  108.757  244.58   
2  2019.11.25 02:15:00  108.758  108.767  108.747  108.762  232.38   
3  2019.11.25 02:20:00  108.763  108.780  108.763  108.771  161.58   
4  2019.11.25 02:25:00  108.772  108.791  108.766  108.774  207.81   

                        Date      MA10       MA20       MA50       EMA10  \
0  2019-11-25 02:05:00+00:00  108.7692  108.76885  108.72808  108.764838   
1  2019-11-25 02:10:00+00:00  108.7657  108.77035  108.72984  108.763413   
2  2019-11-25 02:15:00+00:00  108.7636  108.77080  108.73190  108.763156   
3  2019-11-25 02:20:00+00:00  108.7638  108.77195  108.73382  108.764582   
4  2019-11-25 02:25:00+00:00  108.7634  108.77290  108.73598  108.766294   

        EMA20  Volatility        RSI  Close_lag1  Close_lag2  Close_lag3  \
0  108.760948    0.017110  32.478

NameError: name 'NEWS_OPTION' is not defined

In [ ]:
pip install joblib